In [9]:
import json
from collections import Counter
import string
# Load COCO annotations
def load_coco_annotations(annotation_file):
    with open(annotation_file, 'r') as f:
        annotations = json.load(f)
    return annotations

# Function to preprocess and tokenize a single caption
def preprocess_caption(caption):
    caption = caption.lower().translate(str.maketrans('', '', string.punctuation))
    tokens = caption.split()
    return tokens

# Function to preprocess and tokenize all captions
def preprocess_captions(captions):
    return [preprocess_caption(caption) for caption in captions]

# Function to build a vocabulary from tokenized captions
def build_vocabulary(captions, threshold=5):
    counter = Counter()
    for caption in captions:
        counter.update(caption)

    # Only include words that appear at least 'threshold' times
    words = [word for word, count in counter.items() if count >= threshold]
    word2idx = {word: idx+1 for idx, word in enumerate(words)}

    # Add special tokens
    word2idx['<pad>'] = 0
    word2idx['<start>'] = len(word2idx)
    word2idx['<end>'] = len(word2idx)
    word2idx['<unk>'] = len(word2idx)

    return word2idx

# Function to encode a single caption
def encode_caption(caption, word2idx, max_length):
    encoded = [word2idx.get(word, word2idx['<unk>']) for word in caption]
    encoded = [word2idx['<start>']] + encoded + [word2idx['<end>']]
    encoded = encoded[:max_length]
    while len(encoded) < max_length:
        encoded.append(word2idx['<pad>'])
    return encoded

# Function to encode all captions
def encode_captions(captions, word2idx, max_length):
    return [encode_caption(caption, word2idx, max_length) for caption in captions]

# Paths to annotation files
train_caption_file = '/content/annotwithout/annotations/captions_train2017.json'
val_caption_file = '/content/annotwithout/annotations/captions_val2017.json'

# Load annotations
train_annotations = load_coco_annotations(train_caption_file)
val_annotations = load_coco_annotations(val_caption_file)

# Extract captions
train_captions_list = [ann['caption'] for ann in train_annotations['annotations']]
val_captions_list = [ann['caption'] for ann in val_annotations['annotations']]

# Preprocess captions
processed_train_captions = preprocess_captions(train_captions_list)
processed_val_captions = preprocess_captions(val_captions_list)

# Build vocabulary
vocab = build_vocabulary(processed_train_captions)

# Define maximum caption length
max_length = 20

# Encode captions
encoded_train_captions = encode_captions(processed_train_captions, vocab, max_length)
encoded_val_captions = encode_captions(processed_val_captions, vocab, max_length)

# Print some sample encoded captions
print(encoded_train_captions[:5])
print(encoded_val_captions[:5])


[[10304, 1, 2, 3, 4, 1, 5, 6, 7, 8, 9, 10305, 0, 0, 0, 0, 0, 0, 0, 0], [10304, 1, 10, 4, 11, 12, 13, 1, 14, 15, 13, 16, 10305, 0, 0, 0, 0, 0, 0, 0], [10304, 1, 17, 18, 19, 20, 21, 22, 23, 24, 1, 10306, 22, 17, 10305, 0, 0, 0, 0, 0], [10304, 1, 25, 26, 27, 28, 29, 7, 30, 10305, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [10304, 31, 32, 1, 10306, 33, 34, 35, 36, 1, 37, 38, 39, 10305, 0, 0, 0, 0, 0, 0]]
[[10304, 1, 97, 4178, 149, 22, 36, 8, 119, 1, 584, 10305, 0, 0, 0, 0, 0, 0, 0, 0], [10304, 1, 4178, 149, 22, 36, 1, 164, 90, 10305, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [10304, 58, 173, 1487, 4, 83, 750, 2159, 119, 285, 10305, 0, 0, 0, 0, 0, 0, 0, 0, 0], [10304, 1, 42, 1908, 66, 36, 1, 992, 174, 10305, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [10304, 54, 472, 272, 253, 1, 108, 93, 20, 1, 57, 10305, 0, 0, 0, 0, 0, 0, 0, 0]]


In [10]:
from PIL import Image
import numpy as np
import os

# Function to preprocess a single image
def preprocess_image(image_path, target_size=(256, 256)):
    image = Image.open(image_path).convert('RGB')
    image = image.resize(target_size)
    image = np.array(image) / 255.0
    return image

# Generator function to yield preprocessed images one at a time
def preprocess_images(image_dir, annotations, target_size=(256, 256)):
    image_paths = [os.path.join(image_dir, ann['file_name']) for ann in annotations['images']]
    for image_path in image_paths:
        yield preprocess_image(image_path, target_size)

# Directories containing the images
train_image_dir = '/content/tarin2017without/train2017'
val_image_dir = '/content/val2017without/val2017'

# Using the generator
train_images_gen = preprocess_images(train_image_dir, train_annotations)
val_images_gen = preprocess_images(val_image_dir, val_annotations)

# Example usage: Process images in batches
def process_in_batches(image_gen, batch_size):
    batch = []
    for image in image_gen:
        batch.append(image)
        if len(batch) == batch_size:
            yield np.array(batch)
            batch = []
    if batch:
        yield np.array(batch)

# Example: Processing and storing batches
for train_batch in process_in_batches(train_images_gen, batch_size=32):
    # Process the batch (e.g., store it, use it for training, etc.)
    pass

for val_batch in process_in_batches(val_images_gen, batch_size=32):
    # Process the batch (e.g., store it, use it for validation, etc.)
    pass


In [11]:
import os
import torch
from torch.utils.data import Dataset
from PIL import Image
import numpy as np

class CocoDataset(Dataset):
    def __init__(self, image_dir, annotations, encoded_captions, transform=None):
        self.image_dir = image_dir
        self.annotations = annotations
        self.encoded_captions = encoded_captions
        self.transform = transform
        self.image_paths = [os.path.join(image_dir, ann['file_name']) for ann in annotations['images']]

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image = Image.open(self.image_paths[idx]).convert('RGB')
        if self.transform is not None:
            image = self.transform(image)

        caption = torch.tensor(self.encoded_captions[idx])

        return image, caption

In [12]:
import torch.nn as nn
import torchvision.models as models

class CNNEncoder(nn.Module):
    def __init__(self, embed_size):
        super(CNNEncoder, self).__init__()
        resnet = models.resnet50(pretrained=True)
        for param in resnet.parameters():
            param.requires_grad_(False)

        modules = list(resnet.children())[:-1]
        self.resnet = nn.Sequential(*modules)
        self.linear = nn.Linear(resnet.fc.in_features, embed_size)
        self.bn = nn.BatchNorm1d(embed_size, momentum=0.01)

    def forward(self, images):
        features = self.resnet(images)
        features = features.view(features.size(0), -1)
        features = self.bn(self.linear(features))
        return features


In [13]:
class RNNDecoder(nn.Module):
    def __init__(self, embed_size, hidden_size, vocab_size, num_layers=1):
        super(RNNDecoder, self).__init__()
        self.embed = nn.Embedding(vocab_size, embed_size)
        self.lstm = nn.LSTM(embed_size, hidden_size, num_layers, batch_first=True)
        self.linear = nn.Linear(hidden_size, vocab_size)
        self.init_weights()

    def init_weights(self):
        self.embed.weight.data.uniform_(-0.1, 0.1)
        self.linear.weight.data.uniform_(-0.1, 0.1)
        self.linear.bias.data.fill_(0)

    def forward(self, features, captions):
        embeddings = self.embed(captions)
        embeddings = torch.cat((features.unsqueeze(1), embeddings), 1)
        hiddens, _ = self.lstm(embeddings)
        outputs = self.linear(hiddens)
        return outputs


In [14]:
class ImageCaptioningModel(nn.Module):
    def __init__(self, encoder, decoder):
        super(ImageCaptioningModel, self).__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, images, captions):
        features = self.encoder(images)
        outputs = self.decoder(features, captions)
        return outputs

In [15]:
def custom_collate_fn(data):
    """
    Custom collate function to pad captions.
    Args:
    - data: List of tuples (image, caption).

    Returns:
    - images: Tensor of images.
    - captions: Tensor of padded captions.
    - lengths: List of original lengths of captions.
    """
    # Sort data by caption length (descending order)
    data.sort(key=lambda x: len(x[1]), reverse=True)

    images, captions = zip(*data)

    # Stack images
    images = torch.stack(images, 0)

    # Pad captions
    lengths = [len(cap) for cap in captions]
    padded_captions = torch.zeros(len(captions), max(lengths)).long()
    for i, cap in enumerate(captions):
        end = lengths[i]
        padded_captions[i, :end] = cap[:end]

    return images, padded_captions, lengths

In [16]:
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Dataset
# Hyperparameters
embed_size = 256
hidden_size = 512
vocab_size = len(vocab)
num_layers = 1
learning_rate = 0.001
num_epochs = 10

# Image transformations
transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
])

# Create the datasets
train_dataset = CocoDataset(train_image_dir, train_annotations, encoded_train_captions, transform)
val_dataset = CocoDataset(val_image_dir, val_annotations, encoded_val_captions, transform)

# Create the data loaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2, collate_fn=custom_collate_fn)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2, collate_fn=custom_collate_fn)


In [17]:
import torch.nn.utils.rnn as rnn_utils
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# Initialize the models
encoder = CNNEncoder(embed_size).to(device)
decoder = RNNDecoder(embed_size, hidden_size, vocab_size, num_layers).to(device)
model = ImageCaptioningModel(encoder, decoder).to(device)

# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

# Training the model
for epoch in range(num_epochs):
    model.train()
    for i, (images, captions, lengths) in enumerate(train_loader):
        images = images.to(device)
        captions = captions.to(device)

        outputs = model(images, captions)
        targets = rnn_utils.pack_padded_sequence(captions, lengths, batch_first=True, enforce_sorted=False).data
        outputs = rnn_utils.pack_padded_sequence(outputs, lengths, batch_first=True, enforce_sorted=False).data

        loss = criterion(outputs, targets)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if i % 1000 == 0:
            print(f'Epoch [{epoch}/{num_epochs}], Step [{i}/{len(train_loader)}], Loss: {loss.item():.4f}')

/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth
100%|██████████| 97.8M/97.8M [00:00<00:00, 206MB/s]


Epoch [0/10], Step [0/3697], Loss: 9.2571
Epoch [0/10], Step [1000/3697], Loss: 2.1320
Epoch [0/10], Step [2000/3697], Loss: 1.7350
Epoch [0/10], Step [3000/3697], Loss: 1.7652
Epoch [1/10], Step [0/3697], Loss: 1.8234
Epoch [1/10], Step [1000/3697], Loss: 1.5362
Epoch [1/10], Step [2000/3697], Loss: 1.7665
Epoch [1/10], Step [3000/3697], Loss: 1.6022
Epoch [2/10], Step [0/3697], Loss: 1.5668
Epoch [2/10], Step [1000/3697], Loss: 1.5496
Epoch [2/10], Step [2000/3697], Loss: 1.5949
Epoch [2/10], Step [3000/3697], Loss: 1.4339
Epoch [3/10], Step [0/3697], Loss: 1.3667
Epoch [3/10], Step [1000/3697], Loss: 1.5231
Epoch [3/10], Step [2000/3697], Loss: 1.4336
Epoch [3/10], Step [3000/3697], Loss: 1.4795
Epoch [4/10], Step [0/3697], Loss: 1.3349
Epoch [4/10], Step [1000/3697], Loss: 1.3533
Epoch [4/10], Step [2000/3697], Loss: 1.3667
Epoch [4/10], Step [3000/3697], Loss: 1.4486
Epoch [5/10], Step [0/3697], Loss: 1.2643
Epoch [5/10], Step [1000/3697], Loss: 1.3469
Epoch [5/10], Step [2000/369